# Identity Signal Analysis in Diffusion Model Latent Spaces

> **Hypothesis:** Identity signals occupy specific frequency bands and/or VAE channels in diffusion model latent spaces.

| Model | Resolution | Steps | Guidance |
|-------|-----------|-------|----------|
| SDXL Base 1.0 (fp16) | 1024x1024 | 12 | 7.5 + neg prompt |

In [ ]:
#@title 🔧 GPU Check
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
#@title 📦 Clone/Pull Repo + Install Deps
import os
if os.path.exists('/content/identity-analysis'):
    !cd /content/identity-analysis && git pull
else:
    !git clone https://github.com/JacobWLMS/cosmic-ext-gtk-theme.git /content/identity-analysis
%cd /content/identity-analysis
!pip install -q -r requirements.txt

In [ ]:
#@title 🔌 Imports + Config
import sys, torch, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, HTML
from PIL import Image
from pathlib import Path
sys.path.insert(0, '/content/identity-analysis')

print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#@title 🔑 HF Token + Settings
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("HF token set")
except Exception:
    print("No HF_TOKEN — add via key icon in sidebar if needed")

DEFAULT_STEPS = 12

---
## Exp 7: PCA on Identity
Is identity a linear subspace in latent space? Compare clustering across Ch 3 (fingerprint) vs other channels.

In [ ]:
from experiments.exp7_pca_identity import run as run_exp7
exp7_dir = run_exp7(model_type='sdxl', n_seeds=10, save_latents=False, num_steps=DEFAULT_STEPS)

In [ ]:
# Display Exp 7 plots
import glob, pandas as pd
exp7_path = Path(sorted(glob.glob('outputs/*/experiment_7'))[-1])
print(f"Results in: {exp7_path}")

for p in sorted((exp7_path / 'plots').glob('*.png')):
    print(f'\n{p.name}'); display(Image.open(p))

if (exp7_path / 'summary.csv').exists():
    print("\n=== Channel Comparison Summary ===")
    print(pd.read_csv(exp7_path / 'summary.csv').to_string(index=False))
if (exp7_path / 'results.csv').exists():
    print("\n=== Full Results ===")
    print(pd.read_csv(exp7_path / 'results.csv').to_string(index=False))

---
## Save All Results to Drive

In [ ]:
import shutil, glob, pandas as pd
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

base = Path('/content/drive/MyDrive/identity_analysis/experiments')
exp_map = {
    'experiment_1': 'exp1_paired_frequency',
    'experiment_2': 'exp2_within_identity',
    'experiment_3': 'exp3_identity_emergence',
    'experiment_5': 'exp5_channel_transplant',
    'experiment_6': 'exp6_channel_importance',
    'experiment_7': 'exp7_pca_identity',
}

for exp_dir_name, drive_name in exp_map.items():
    dirs = sorted(glob.glob(f'outputs/*/{exp_dir_name}'))
    if not dirs:
        continue
    exp_dir = Path(dirs[-1])
    drive_dir = base / drive_name
    for d in ['data', 'plots', 'samples']:
        (drive_dir / d).mkdir(parents=True, exist_ok=True)
    saved = 0
    for f in exp_dir.glob('*.csv'):
        shutil.copy2(f, drive_dir / 'data' / f.name); saved += 1
    if (exp_dir / 'plots').exists():
        for f in (exp_dir / 'plots').glob('*.png'):
            is_sample = any(k in f.name for k in ['identity_a', 'identity_b', 'sample_', 'original', 'zeroed', 'final_', 'source', 'target'])
            shutil.copy2(f, drive_dir / ('samples' if is_sample else 'plots') / f.name); saved += 1
    print(f"{drive_name}: {saved} files saved")